# 🚀 Aluminum Defect Detection - YOLOv8 Training on Google Colab (GPU)

This notebook trains the **YOLOv8** model for 11-class aluminum profile defect classification using a free Google Colab GPU (T4 / V100 / A100).

> **Step 0: Enable GPU Hardware Acceleration**  
> In the top menu, go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.

In [18]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("renshuddd/ali2018-classfication")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ali2018-classfication' dataset.
Path to dataset files: /kaggle/input/ali2018-classfication


In [21]:
# Copy the dataset from read-only /kaggle/ directory to a writable /content/ location
!mkdir -p /content/data_cls
!cp -r /kaggle/input/ali2018-classfication/ali2018/* /content/data_cls/
print("✓ Dataset copied to writable location /content/data_cls")

✓ Dataset copied to writable location /content/data_cls


In [1]:
# 1. Verify GPU Allocation
!nvidia-smi

Sat Oct 17 17:24:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 2. Clone the GitHub Repository
Clone your repository to get all scripts, configs, and utilities.

In [2]:
!git clone https://github.com/Sanskar-Shukla8303/Aluminum_defect.git
%cd /content/Aluminum_defect/aluminum_defect_yolo
!pwd

Cloning into 'Aluminum_defect'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 44 (delta 2), reused 5 (delta 2), pack-reused 35 (from 1)
Receiving objects: 100% (44/44), 46.42 MiB | 32.58 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/Aluminum_defect/aluminum_defect_yolo
/content/Aluminum_defect/aluminum_defect_yolo


### 3. Install Required Dependencies
Install Ultralytics and dependencies.

In [3]:
!pip install -q ultralytics scikit-learn matplotlib seaborn pyyaml

import ultralytics
ultralytics.checks()

Ultralytics 8.4.180 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 42.5/112.6 GB disk)


### 4. Provide the Dataset (`data_cls`)

**Option A: Mount Google Drive (Recommended if `data_cls.zip` is in your Drive)**
Upload your `data_cls` zip folder to your Google Drive, then run the cell below:

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# If data_cls.zip is in MyDrive:
# !unzip -q /content/drive/MyDrive/data_cls.zip -d /content/Aluminum_defect/aluminum_defect_yolo/
# print('✓ Dataset unzipped successfully!')

Mounted at /content/drive


**Option B: Upload `data_cls.zip` directly to Colab**  
Click the folder icon 📁 on the left sidebar in Colab, drag & drop `data_cls.zip` into `/content/Aluminum_defect/aluminum_defect_yolo/`, then run:

In [13]:
# Try unzipping the alternative dataset zip file
!unzip -q /content/drive/MyDrive/aluminum_defect_yolo/scripts/datasets.zip -d /content/Aluminum_defect/aluminum_defect_yolo/

import os
if os.path.exists('/content/Aluminum_defect/aluminum_defect_yolo/data_cls'):
    print('✓ Dataset successfully extracted and ready at /content/Aluminum_defect/aluminum_defect_yolo/data_cls')
else:
    print('❌ Failed to extract data_cls from datasets.zip. Your zip files might be corrupted or incomplete.')

[/content/drive/MyDrive/aluminum_defect_yolo/scripts/datasets.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/drive/MyDrive/aluminum_defect_yolo/scripts/datasets.zip or
        /content/drive/MyDrive/aluminum_defect_yolo/scripts/datasets.zip.zip, and cannot find /content/drive/MyDrive/aluminum_defect_yolo/scripts/datasets.zip.ZIP, period.
❌ Failed to extract data_cls from datasets.zip. Your zip files might be corrupted or incomplete.


In [14]:
import os

def search_folders(path):
    for root, dirs, files in os.walk(path):
        # Look for folders with train/val structure characteristic of classification datasets
        if 'train' in dirs and 'val' in dirs:
            print(f"✓ Found potential classification dataset directory at: {root}")
            print("Subdirectories inside:", dirs)
            return root
    return None

search_folders('/content/Aluminum_defect')

In [19]:
import os
alt_drive_path = '/kaggle/input/ali2018-classfication'
if os.path.exists(alt_drive_path):
    print(f"Contents of {alt_drive_path}:")
    for root, dirs, files in os.walk(alt_drive_path):
        # limit print depth
        depth = root.replace(alt_drive_path, '').count(os.sep)
        if depth < 2:
            print(f"  {'  ' * depth}📁 {os.path.basename(root)}/ contains {len(dirs)} folders, {len(files)} files")
else:
    print(f"Directory {alt_drive_path} does not exist.")

Contents of /kaggle/input/ali2018-classfication:
  📁 ali2018-classfication/ contains 1 folders, 0 files
    📁 ali2018/ contains 11 folders, 0 files


In [16]:
# Download a clean and verified version of the aluminum defect dataset
import os
import urllib.request

dataset_url = 'https://github.com/Sanskar-Shukla8303/Aluminum_defect/releases/download/v1.0/data_cls.zip'
dest_zip = '/content/Aluminum_defect/aluminum_defect_yolo/data_cls.zip'

try:
    print("Downloading clean dataset from repository releases...")
    urllib.request.urlretrieve(dataset_url, dest_zip)
    print("✓ Download complete. Extracting dataset...")
    !unzip -q {dest_zip} -d /content/Aluminum_defect/aluminum_defect_yolo/
    if os.path.exists('/content/Aluminum_defect/aluminum_defect_yolo/data_cls'):
        print('✓ Dataset successfully extracted and ready at /content/Aluminum_defect/aluminum_defect_yolo/data_cls!')
    else:
        print('❌ Extraction completed but "data_cls" directory is missing.')
except Exception as e:
    print(f"Could not download from release asset: {e}")
    print("\n👉 ACTION REQUIRED: Please manually upload a fresh, uncorrupted 'data_cls.zip' file directly into the Colab files sidebar on the left, then run the extraction step.")

Could not download from release asset: HTTP Error 404: Not Found

👉 ACTION REQUIRED: Please manually upload a fresh, uncorrupted 'data_cls.zip' file directly into the Colab files sidebar on the left, then run the extraction step.


In [9]:
import os
# Check the contents of the current directory and the cloned repository
print("Current Working Directory:", os.getcwd())
print("Contents of Current Directory:", os.listdir('.'))

# Let's see if there is any 'data_cls' directory inside Aluminum_defect
for root, dirs, files in os.walk('/content/Aluminum_defect'):
    if 'data_cls' in dirs or 'data_cls.zip' in files:
        print(f"\n✓ Found dataset reference at: {root}")
        print("Subdirectories/files inside:", os.listdir(os.path.join(root, 'data_cls')) if 'data_cls' in dirs else files)

Current Working Directory: /content/Aluminum_defect/aluminum_defect_yolo
Contents of Current Directory: ['Colab_Connection.ipynb', '.gitignore', 'COMPLETE.md', 'yolov8m-cls.pt', 'SETUP_SUMMARY.md', 'TROUBLESHOOTING.md', 'config.yaml', 'project_structure.txt', 'images_4.jpg', 'Colab_Training.ipynb', 'QUICKSTART.md', 'ss.ipynb', 'FILES_INVENTORY.md', 'INDEX.md', 'README.md', 'WELCOME.txt', 'START_HERE.md', 'runs', 'requirements.txt', 'data', 'PROJECT_OVERVIEW.md', 'scripts', 'results']


In [10]:
import os
data_path = '/content/Aluminum_defect/aluminum_defect_yolo/data'
if os.path.exists(data_path):
    print(f"Contents of {data_path}:", os.listdir(data_path))
    # Check if there are subfolders like train/val inside
    for sub in os.listdir(data_path):
        sub_path = os.path.join(data_path, sub)
        if os.path.isdir(sub_path):
            print(f"  -> {sub}/ contains:", os.listdir(sub_path)[:5])
else:
    print("The directory 'data' does not exist.")

Contents of /content/Aluminum_defect/aluminum_defect_yolo/data: ['dataset.yaml']


In [11]:
import os
# Let's search your Google Drive to see if there is any dataset file containing 'data_cls' or 'defect'
drive_path = '/content/drive/MyDrive'
if os.path.exists(drive_path):
    print("Searching Google Drive for datasets...")
    found_files = []
    for root, dirs, files in os.walk(drive_path):
        for file in files:
            if 'data_cls' in file.lower() or 'defect' in file.lower() or file.endswith('.zip'):
                # Restrict depth or print relevant items
                full_path = os.path.join(root, file)
                if 'colab notebook' not in root.lower() and len(found_files) < 15:
                    found_files.append(full_path)
    if found_files:
        print("\nPossible datasets found in Google Drive:")
        for f in found_files:
            print(f" - {f}")
    else:
        print("No matching zip or dataset files found in Google Drive. Please upload 'data_cls.zip' directly to the files sidebar or Google Drive.")
else:
    print("Google Drive is not mounted or accessible.")

Searching Google Drive for datasets...

Possible datasets found in Google Drive:
 - /content/drive/MyDrive/aluminum_defect_yolo/data_cls.zip
 - /content/drive/MyDrive/aluminum_defect_yolo/scripts/datasets.zip


### 5. Train YOLOv8 on GPU

You can choose the model size: `yolov8n-cls` (fastest), `yolov8s-cls` (balanced), or `yolov8m-cls` (high accuracy).

In [ ]:
# Run training using the writable /content/data_cls directory
!python scripts/train.py \
    --model yolov8m-cls \
    --epochs 50 \
    --batch 32 \
    --imgsz 640 \
    --data /content/data_cls

✓ Using GPU: Tesla T4
  CUDA Version: 13.0

ALUMINUM DEFECT DETECTION - YOLO TRAINING

→ Loading model: yolov8m-cls

→ Training configuration:
  Model: yolov8m-cls
  Dataset: /content/data_cls
  Epochs: 50
  Batch size: 32
  Image size: 640
  Device: 0
  Results directory: results/run_20261017_174203

→ Starting training...
Ultralytics 8.4.180 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_cls, degrees=10, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchsc

In [20]:
# Run training using the downloaded Kaggle dataset path
!python scripts/train.py \
    --model yolov8m-cls \
    --epochs 50 \
    --batch 32 \
    --imgsz 640 \
    --data /kaggle/input/ali2018-classfication/ali2018

✓ Using GPU: Tesla T4
  CUDA Version: 13.0

ALUMINUM DEFECT DETECTION - YOLO TRAINING

→ Loading model: yolov8m-cls

→ Training configuration:
  Model: yolov8m-cls
  Dataset: /kaggle/input/ali2018-classfication/ali2018
  Epochs: 50
  Batch size: 32
  Image size: 640
  Device: 0
  Results directory: results/run_20261017_174115

→ Starting training...
Ultralytics 8.4.180 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/ali2018-classfication/ali2018, degrees=10, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, ex

### 6. Test Model with Inference

In [7]:
import glob
import os

# Locate the latest trained best.pt model
trained_models = glob.glob('results/**/weights/best.pt', recursive=True) + glob.glob('runs/**/weights/best.pt', recursive=True)
if trained_models:
    latest_model = trained_models[-1]
    print(f'Using model: {latest_model}')
    !python scripts/inference.py --model {latest_model} --source images_4.jpg
else:
    print('No trained weights found yet.')

No trained weights found yet.


### 7. Download or Save Trained Model (`best.pt`)
Save the best weights back to Google Drive or download directly to your PC.

In [8]:
from google.colab import files
import glob

trained_models = glob.glob('results/**/weights/best.pt', recursive=True) + glob.glob('runs/**/weights/best.pt', recursive=True)
if trained_models:
    best_weight = trained_models[-1]
    print(f'Downloading {best_weight}...')
    files.download(best_weight)

    # Optional: copy to Google Drive
    # !cp {best_weight} /content/drive/MyDrive/aluminum_best_yolov8m.pt
else:
    print('No weights to download.')

No weights to download.
